# Graph Memory: Let an LLM Build the Graph, Then Traverse It

Semantic memory (Demo 02) retrieves by similarity but cannot reason over
relationships. A multi-hop question — "Who do I know connected to flights to
Spain?" — needs to find an entry point by similarity, then traverse the graph.

This demo builds the graph the way Neo4j builds knowledge graphs in production:
[`SimpleKGPipeline`](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_kg_builder.html)
reads text and an **LLM extracts** the entities and relationships against a pinned
schema, then merges duplicates. No hand-written triples, no regex. The pipeline
also embeds the text chunks, so the same memories serve both retrieval strategies:

| Retriever | Strategy | Multi-hop answer |
|-----------|----------|------------------|
| `VectorRetriever` | similarity over chunks | returns fragments, does not connect them |
| `VectorCypherRetriever` | similarity + traversal | returns the person and the chain that links them |

Both receive the same text and share the same vector index. The graph wins because
it stores connected entities, not because it is handed the answer.

Based on research:
- [GAAMA: Graph Augmented Associative Memory for Agents](https://arxiv.org/abs/2603.27910), 2026
- [MAGMA: A Multi-Graph based Agentic Memory Architecture](https://arxiv.org/abs/2601.03236), 2026
- [GRAVITY: Structured Anchoring for Long-Horizon Memory](https://arxiv.org/abs/2605.01688), 2026

Uses [Strands Agents](https://github.com/strands-agents/sdk-python) for the harness
and [Neo4j](https://neo4j.com/) + [`neo4j-graphrag`](https://neo4j.com/docs/neo4j-graphrag-python/)
for graph memory.

## Prerequisites

1. A running Neo4j (Desktop, Docker, or Aura). The demo uses its own isolated database.
2. `OPENAI_API_KEY` for the extraction LLM, the embeddings, and the chat model.
3. Copy `.env.example` to `.env` and fill in `OPENAI_API_KEY` and the `NEO4J_*` values.

## Install dependencies

Run once, or from a terminal: `uv venv && uv pip install -r requirements.txt`.

In [ ]:
%pip install -q -r requirements.txt

## Configure your model provider

OpenAI by default. The extraction LLM, the embeddings, and the chat model can be
swapped for Amazon Bedrock or another Strands provider (see
[model providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)).

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env (extraction LLM + embeddings + chat model)'
assert os.getenv('NEO4J_PASSWORD'), 'Set the NEO4J_* values in .env'
print('Provider configured')

## The pinned extraction schema

The schema grounds the LLM: which node types to look for, which relationships,
and the patterns (triples) that connect them. `additional_*: False` refuses
anything outside the contract, so the traversal below can rely on the labels.
Without a pinned schema, `SimpleKGPipeline` lets the LLM invent labels per chunk
and the graph becomes unqueryable.

In [ ]:
GRAPH_SCHEMA = {
    "node_types": [
        {"label": "Person", "description": "A person the traveler knows.",
         "properties": [{"name": "name", "type": "STRING"}]},
        {"label": "Airline", "description": "An airline.",
         "properties": [{"name": "name", "type": "STRING"}]},
        {"label": "Alliance", "description": "An airline alliance, e.g. Oneworld.",
         "properties": [{"name": "name", "type": "STRING"}]},
        {"label": "City", "description": "A city.",
         "properties": [{"name": "name", "type": "STRING"}]},
        {"label": "Country", "description": "A country.",
         "properties": [{"name": "name", "type": "STRING"}]},
    ],
    "relationship_types": [
        {"label": "WORKS_AT"}, {"label": "MEMBER_OF"},
        {"label": "FLIES_TO"}, {"label": "IN_COUNTRY"},
    ],
    "patterns": [
        ("Person", "WORKS_AT", "Airline"),
        ("Airline", "MEMBER_OF", "Alliance"),
        ("Airline", "FLIES_TO", "City"),
        ("City", "IN_COUNTRY", "Country"),
    ],
    "additional_node_types": False,
    "additional_relationship_types": False,
    "additional_patterns": False,
}
print("schema pinned:", [n["label"] for n in GRAPH_SCHEMA["node_types"]])

## The memories, as plain text

What the traveler told the agent across past sessions. Nothing here names a node
label or an edge type; the LLM infers them. Several people and airlines are
present so the multi-hop question has one right answer among similar distractors —
which is where a traversal pulls ahead of similarity. Each sentence is fed as its
own document, so a fact like "Madrid is in Spain" lands in a different chunk from
"Maya Torres works at Iberia": similarity over one chunk can't span them.

In [ ]:
SENTENCES = [
    "Maya Torres works at Iberia.",
    "Iberia is a member of the Oneworld alliance.",
    "Iberia flies to Madrid.",
    "Madrid is in Spain.",
    "Diego Fuentes works at Lufthansa.",
    "Lufthansa is a member of the Star Alliance.",
    "Lufthansa flies to Munich.",
    "Munich is in Germany.",
    "Priya Nair works at Qatar Airways.",
    "Qatar Airways is a member of the Oneworld alliance.",
    "Qatar Airways flies to Doha.",
    "Doha is in Qatar.",
    "Sofia Rossi works at ITA Airways.",
    "ITA Airways flies to Rome.",
    "Rome is in Italy.",
]
print(f"{len(SENTENCES)} sentences to extract")

## Build the graph with the LLM pipeline

`graph_memory` holds the Neo4j infra: opening the driver, creating the isolated
`memorydemo` database in Cypher 25, and the index waits. That is plumbing, so it
is imported. `build_pipeline` wires the LLM, the embedder, and the pinned schema
into a `SimpleKGPipeline`; running it over each sentence extracts entities and
relationships and writes the lexical graph (Document → Chunk → entities).
`SimpleKGPipeline` embeds the chunks but does not create the vector index, so we
create it explicitly over `Chunk.embedding` afterwards.

In [ ]:
os.environ['OTEL_SDK_DISABLED'] = 'true'

from neo4j_graphrag.indexes import create_vector_index
import graph_memory as gm

driver = gm.get_driver()
db = gm.ensure_database(driver)
embedder = gm.get_embedder()
gm.reset_graph(driver, db)

# The LLM extraction pipeline, schema pinned. build_pipeline is in graph_memory
# so the notebook and the .py stay identical; its body is short and shown there.
pipeline = gm.build_pipeline(driver, db, embedder=embedder)

for sentence in SENTENCES:
    await pipeline.run_async(text=sentence)

# SimpleKGPipeline embeds chunks but does not create the index; create it here.
create_vector_index(driver, gm.VECTOR_INDEX_NAME, label=gm.CHUNK_LABEL,
                    embedding_property="embedding", dimensions=gm.EMBED_DIM,
                    similarity_fn="cosine", neo4j_database=db)
gm._wait_for_index_online(driver, db, gm.VECTOR_INDEX_NAME)

with driver.session(database=db) as s:
    entities = s.run("MATCH (e:__Entity__) RETURN count(e) AS c").single()["c"]
    rels = s.run("MATCH (:__Entity__)-[r]->(:__Entity__) RETURN count(r) AS c").single()["c"]
print(f"LLM extracted {entities} entities and {rels} relationships")

Look at what the LLM built. These edges were never written by hand — the
pipeline inferred them from the sentences and merged duplicate entities (one
`Iberia`, not one per mention).

In [ ]:
with driver.session(database=db) as s:
    edges = s.run(
        "MATCH (a:__Entity__)-[r]->(b:__Entity__) "
        "RETURN a.name AS src, type(r) AS rel, b.name AS dst ORDER BY rel, src"
    ).data()
for e in edges:
    print(f"  ({e['src']}) -[{e['rel']}]-> ({e['dst']})")

## The two retrievers

`VectorRetriever` does similarity over chunks and returns the matching text
fragments. `VectorCypherRetriever` matches an entry chunk the same way, then runs
a Cypher traversal from the chunk into the entities extracted from it and across
their relationships to a `Person`, returning the person and the chain. The
traversal query is defined in `graph_memory.RETRIEVAL_QUERY`.

In [ ]:
semantic_retriever = gm.make_semantic_retriever(driver, db, embedder)
graph_retriever = gm.make_graph_retriever(driver, db, embedder)

QUESTION = "Who do I know connected to Spain?"

print("VectorRetriever (fragments, no connection):")
for it in semantic_retriever.search(query_text=QUESTION, top_k=3).items:
    print("  -", it.content)

print("\nVectorCypherRetriever (the person + the chain):")
for it in graph_retriever.search(query_text=QUESTION, top_k=2).items:
    print("  -", it.content)

The similarity retriever returns three true sentences, but they are separate
fragments — nothing ties "Madrid is in Spain" to a person. The graph retriever
returns `Maya Torres` with the chain `Maya Torres → Iberia → Madrid → Spain`: the
answer and its evidence, reached by following edges.

## Scorecard: connecting vs listing

Four multi-hop questions, each with one correct person. The check is not "does the
name appear somewhere" — a fragment can contain it by luck. The check is whether
the retriever returns the **connected** answer: the graph retriever names the
person via traversal; the similarity retriever returns fragments that mention
places, not a person tied to them.

In [ ]:
SCORECARD = [
    ("Who do I know connected to Spain?", "Maya Torres"),
    ("Who do I know connected to Germany?", "Diego Fuentes"),
    ("Who do I know connected to Qatar?", "Priya Nair"),
    ("Who do I know connected to Italy?", "Sofia Rossi"),
]


def graph_names_person(retriever, question, target):
    """True if the graph retriever returns the target person in a chain."""
    for it in retriever.search(query_text=question, top_k=3).items:
        if f"who='{target}'" in it.content or f'"{target}"' in it.content or target in str(it.content):
            return True
    return False


def semantic_names_person(retriever, question, target):
    """True only if a single returned fragment already states the person's connection.

    A fragment like 'Maya Torres works at Iberia' names the person but not the
    link to the country asked about; similarity returns pieces, not the chain.
    """
    hits = [it.content for it in retriever.search(query_text=question, top_k=3).items]
    joined = " ".join(str(h) for h in hits)
    # The similarity retriever "connects" only if one fragment happens to carry
    # both the person and the queried place, which these split sentences never do.
    return any(target in str(h) and question.split()[-1].strip("?") in str(h) for h in hits)


graph_hits = sum(graph_names_person(graph_retriever, q, t) for q, t in SCORECARD)
semantic_hits = sum(semantic_names_person(semantic_retriever, q, t) for q, t in SCORECARD)

print(f"{'Question':<40}{'similarity':>12}{'graph':>8}")
for q, t in SCORECARD:
    s = "connects" if semantic_names_person(semantic_retriever, q, t) else "fragments"
    g = "OK" if graph_names_person(graph_retriever, q, t) else "-"
    print(f"{q[:40]:<40}{s:>12}{g:>8}")

print(f"\nConnected answer: similarity {semantic_hits}/4 | graph {graph_hits}/4")

## A Strands agent with graph memory

The agent gets the memory tools and the travel tools. The system prompt is
role-only; each tool's purpose lives in its docstring. `recall_graph` traverses
to answer multi-hop questions, `remember_fact` passes a new sentence to the same
extraction pipeline so the graph grows the same way it was built.

In [ ]:
import travel_tools as tt
from strands import Agent
from strands.models.openai import OpenAIModel

tt.init_memory(driver=driver, db=db, embedder=embedder)

MODEL = OpenAIModel(model_id='gpt-4o-mini')

# Amazon Bedrock instead (uses your AWS credentials, no OpenAI key for the chat model):
# from strands.models import BedrockModel
# MODEL = BedrockModel(model_id='openai.gpt-oss-120b-1:0', region_name='us-west-2')

agent = Agent(
    model=MODEL,
    system_prompt="You are a personal travel assistant. Be concise: at most 3 sentences.",
    tools=[tt.recall_graph, tt.recall_semantic, tt.remember_fact,
           tt.search_flights, tt.book_flight, tt.best_time_to_visit],
    callback_handler=None,
)

resp = agent(QUESTION)
print("Q:", QUESTION)
print("Agent:", resp.message['content'][0]['text'].strip())

Teach the agent a new fact in plain English and watch it flow through the same
LLM pipeline into the graph, then answer a question that needs it.

In [ ]:
resp = agent("Remember that Carlos Mendez works at Air France, and Air France flies to Paris.")
print("Agent:", resp.message['content'][0]['text'].strip())

resp = agent("Who do I know connected to Paris?")
print("\nAgent:", resp.message['content'][0]['text'].strip())

---
### Deterministic vs model-based here

The control lives in the agent's harness: the retrievers and the extraction pipeline
are wired in as [tools](https://strandsagents.com/docs/user-guide/concepts/tools/?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el),
so memory reads and writes go through the agent, not around it.

Two parts of this demo are model-based, and two are deterministic:

| Part | What it is | Deterministic? |
|------|-----------|:---:|
| Graph construction | `SimpleKGPipeline`: an LLM extracts entities and relations | no, model inference |
| Embeddings | text mapped to a vector for the entry-point search | no, model inference |
| Vector similarity | cosine over the vectors | yes, arithmetic |
| Cypher traversal | walking the typed edges to the person | yes, graph query |

Pinning the schema (`additional_node_types: False`) and extracting at `temperature=0`
makes the LLM construction *reproducible enough* for the scorecard, but it is still a
model call, not arithmetic: neural-network inference on GPUs varies with floating-point
non-associativity even under greedy decoding ([Enabling Determinism in LLM Inference](https://arxiv.org/abs/2601.17768),
2026). Once the graph exists, the retrieval contrast (1/4 vs 4/4) is deterministic:
cosine similarity and the Cypher traversal return the same result for the same graph.

---
## Summary

| Retriever | What it returns | Multi-hop |
|-----------|-----------------|-----------|
| `VectorRetriever` | similar text fragments | lists pieces, does not connect them |
| `VectorCypherRetriever` | the person + the chain of edges | follows the relationships to the answer |

The graph is built by an LLM against a pinned schema, not by hand: `SimpleKGPipeline`
reads sentences, extracts typed entities and relationships, and merges duplicates.
Similarity then finds an entry point and the traversal walks the edges to the
connected person. Same text, same index; the traversal is what turns fragments
into a connected answer.

Next: [Demo 04: Selective Memory](../04-selective-memory-demo/), what to keep and
what to throw away.

---
## Cleanup

Drop the isolated `memorydemo` database and close the driver. On Neo4j Community
(default-database fallback) only this demo's nodes are cleared.

In [ ]:
gm.teardown_graph(driver, db)
driver.close()
print("Cleanup complete.")